# Copy an archive from S3 to Google Drive

Copies one archive from Account A's S3 bucket into Google Drive. `COPY_WHAT` in §1 picks which:

| `COPY_WHAT` | S3 source (`s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/…`) | Size | Drive destination |
|---|---|---|---|
| `clean12_augmented` (default) | `clean12_augmented/clean_ecg_signal_images_clean12_augmented.tar`, the output of `Clean12_Augmented_ECG_Images_EC2_AccountA.ipynb` | ~160 GiB | `MyDrive/CardioSentry/new_dataset/`, where the clean12 + augmented training notebook reads it |
| `clean12_signals` | `clean12/ECG_500Hz_clean12.tar.gz`, the 113,358 native-500 Hz WFDB records | ~5.5 GB | `MyDrive/CardioSentry/datasets/` |

Next to the archive it writes `<archive>.sha256` (the S3 sidecar, unchanged) and `<archive>.source.json` (S3 URI, size, ETag, SHA-256).
- **No local copy.** S3 byte ranges go straight into a resumable Drive upload, so nothing is stored on the Colab disk.
- **Verified.** Drive's own SHA-256 of the stored file must equal the S3 sidecar. If it does not, the copy is deleted.
- **Cost.** One S3 download of the archive out of AWS, at S3's internet data-transfer rate (about USD 0.09/GB: ~USD 15 for the augmented archive, ~USD 0.50 for the signals). Writing to Drive is free within your storage quota.
- **Safe to rerun.** If the verified archive is already on Drive, nothing is copied again. A copy cut off by a runtime disconnect starts again from the beginning.

**Before you run**
1. The destination folder must exist (or set `DRIVE_DEST_DIR` / `DRIVE_DEST_FOLDER_ID` in §1), and the Drive account needs the archive's size + 1 GiB free.
2. Colab **Secrets** (key icon, notebook access on): `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` of an IAM user in Account A with `s3:GetObject` on `CardioSentry/dataset/clean12_augmented/*` (and on `CardioSentry/dataset/clean12/*` for the signals).
3. A CPU runtime is enough. Runtime > Run all, and sign in at both Google prompts (Drive mount, then Drive API) with the same account. Keep the tab open until §3 prints `Done`.

In [ ]:
# 1. Configuration
import os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Which archive to copy (table in the top cell).
COPY_WHAT = "clean12_augmented"
_S3_DATASET = "s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/"
S3_SOURCE, DRIVE_DEST_DIR = {
    "clean12_augmented": (_S3_DATASET + "clean12_augmented/clean_ecg_signal_images_clean12_augmented.tar",
                          "/content/drive/MyDrive/CardioSentry/new_dataset"),
    "clean12_signals": (_S3_DATASET + "clean12/ECG_500Hz_clean12.tar.gz",
                        "/content/drive/MyDrive/CardioSentry/datasets"),
}[COPY_WHAT]
S3_SHA256_SOURCE = S3_SOURCE + ".sha256"
ARCHIVE_NAME = S3_SOURCE.rsplit("/", 1)[1]

DRIVE_DEST_FOLDER_ID = ""        # optional: the folder's id from its URL; skips the path lookup
DRIVE_API = "https://www.googleapis.com"
DRIVE_CHUNK_BYTES = 64 * 2**20   # resumable-upload chunk; a multiple of 256 KiB
assert DRIVE_CHUNK_BYTES % (256 * 1024) == 0
print("Copy", S3_SOURCE, "->", f"{DRIVE_DEST_DIR}/{ARCHIVE_NAME}")

In [ ]:
%pip install --quiet boto3 google-auth requests
import hashlib
import json
import time
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3

## 2. Google Drive and AWS access

Mounts Drive and signs in to the Drive API (same account), resolves `DRIVE_DEST_DIR` to its Drive id, and copies the AWS read key from Colab Secrets. These are the same helpers the augmentation notebook uses.

In [ ]:
import time
import requests

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"
DRIVE_FILE_FIELDS = "id,name,mimeType,size,md5Checksum,sha256Checksum,shortcutDetails,capabilities/canAddChildren"


class DriveTransientError(Exception):
    pass


class DriveSessionExpired(Exception):
    pass


def drive_authorized_session():
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    if IN_COLAB:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
    return AuthorizedSession(credentials)


def drive_request(method, path, ok=(200,), tries=8, **kwargs):
    # One Drive API call; retries rate limits, server errors and dropped connections.
    url = path if path.startswith("http") else DRIVE_API + path
    error = None
    for attempt in range(tries):
        try:
            response = drive_http.request(method, url, timeout=(30, 600), **kwargs)
        except requests.RequestException as exc:
            error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code in ok:
                return response
            retryable = response.status_code in (408, 429, 500, 502, 503, 504) or (
                response.status_code == 403 and "ateLimitExceeded" in response.text)
            if not retryable:
                raise RuntimeError(f"Drive API {method} {url.split('?')[0]} -> {response.status_code}: "
                                   f"{response.text[:500]}")
            error = f"{response.status_code}: {response.text[:200]}"
        time.sleep(min(60, 2 ** attempt))
    raise RuntimeError(f"Drive API {method} {url.split('?')[0]} failed {tries} times; last: {error}")


def drive_children(folder_id, name=None):
    query = f"'{folder_id}' in parents and trashed=false"
    if name is not None:
        query += " and name='" + name.replace("\\", "\\\\").replace("'", "\\'") + "'"
    files, token = [], None
    while True:
        params = {"q": query, "fields": f"nextPageToken,files({DRIVE_FILE_FIELDS})", "pageSize": 1000,
                  "supportsAllDrives": "true", "includeItemsFromAllDrives": "true"}
        if token:
            params["pageToken"] = token
        page = drive_request("GET", "/drive/v3/files", params=params).json()
        files += page.get("files", [])
        token = page.get("nextPageToken")
        if not token:
            return files


def drive_meta(file_id):
    response = drive_request("GET", f"/drive/v3/files/{file_id}", ok=(200, 404),
                             params={"fields": DRIVE_FILE_FIELDS, "supportsAllDrives": "true"})
    return response.json() if response.status_code == 200 else None


def drive_download(file_id, start=None, length=None):
    headers = {"Range": f"bytes={start}-{start + length - 1}"} if start is not None else {}
    data = drive_request("GET", f"/drive/v3/files/{file_id}", ok=(200, 206), headers=headers,
                         params={"alt": "media", "supportsAllDrives": "true"}).content
    if length is not None and len(data) != length:
        raise DriveTransientError(f"ranged download returned {len(data)} of {length} bytes")
    return data


def drive_delete(file_id):
    drive_request("DELETE", f"/drive/v3/files/{file_id}", ok=(200, 204, 404),
                  params={"supportsAllDrives": "true"})


def drive_folder(parent_id, name):
    # The one folder called name in parent_id, created if missing.
    hits = [f for f in drive_children(parent_id, name) if f["mimeType"] == FOLDER_MIME]
    if len(hits) > 1:
        raise RuntimeError(f"{len(hits)} folders named {name!r} in Drive folder {parent_id}; keep one")
    if hits:
        return hits[0]["id"]
    return drive_request("POST", "/drive/v3/files", params={"supportsAllDrives": "true", "fields": "id"},
                         json={"name": name, "mimeType": FOLDER_MIME, "parents": [parent_id]}).json()["id"]


def drive_free_bytes():
    quota = drive_request("GET", "/drive/v3/about", params={"fields": "storageQuota"}).json()["storageQuota"]
    return None if "limit" not in quota else int(quota["limit"]) - int(quota["usage"])


def drive_resumable_start(parent_id, name, size=None):
    # A resumable upload session (valid for a week); size=None leaves the length open.
    headers = {"X-Upload-Content-Type": "application/octet-stream"}
    if size is not None:
        headers["X-Upload-Content-Length"] = str(size)
    response = drive_request("POST", "/upload/drive/v3/files", headers=headers,
                             params={"uploadType": "resumable", "supportsAllDrives": "true",
                                     "fields": DRIVE_FILE_FIELDS},
                             json={"name": name, "parents": [parent_id]})
    return response.headers["Location"]


def _committed(response):
    # Bytes Drive holds after a 308: "Range: bytes=0-N" means N + 1; no header means 0.
    value = response.headers.get("Range")
    return int(value.rsplit("-", 1)[1]) + 1 if value else 0


def drive_resumable_status(uri, total, declared=True):
    # (bytes Drive holds, file metadata if the upload is already complete).
    for attempt in range(12):
        try:
            response = drive_http.put(uri, data=b"", timeout=(30, 120),
                                      headers={"Content-Range": f"bytes */{total if declared else '*'}"})
        except requests.RequestException:
            time.sleep(min(60, 2 ** attempt))
            continue
        if response.status_code in (200, 201):
            return total, response.json()
        if response.status_code == 308:
            return _committed(response), None
        if response.status_code in (404, 410):
            raise DriveSessionExpired(f"upload session expired ({response.status_code})")
        if response.status_code not in (408, 429, 500, 502, 503, 504):
            raise RuntimeError(f"upload status -> {response.status_code}: {response.text[:300]}")
        time.sleep(min(60, 2 ** attempt))
    raise RuntimeError("upload status query kept failing")


def drive_resumable_upload(uri, total, read, declared=True, on_commit=None):
    # Send bytes [committed, total) in chunks; read(offset, n) supplies them. Returns file metadata.
    committed, meta = drive_resumable_status(uri, total, declared)
    failures = 0
    while meta is None:
        if on_commit:
            on_commit(committed)
        start = committed
        data = read(start, min(DRIVE_CHUNK_BYTES, total - start))
        end = start + len(data)
        size_field = total if declared or end == total else "*"
        try:
            response = drive_http.put(uri, data=data, timeout=(30, 900),
                                      headers={"Content-Range": f"bytes {start}-{end - 1}/{size_field}"})
            if response.status_code in (200, 201):
                committed, meta = total, response.json()
            elif response.status_code == 308:
                committed, failures = _committed(response), 0
            elif response.status_code in (404, 410):
                raise DriveSessionExpired(f"upload session expired ({response.status_code})")
            elif response.status_code in (408, 429, 500, 502, 503, 504):
                raise DriveTransientError(f"{response.status_code}: {response.text[:200]}")
            else:
                raise RuntimeError(f"upload chunk -> {response.status_code}: {response.text[:300]}")
        except (requests.RequestException, DriveTransientError) as exc:
            failures += 1
            if failures > 12:
                raise RuntimeError(f"upload kept failing at byte {start:,}: {exc}") from exc
            time.sleep(min(60, 2 ** failures))
            committed, meta = drive_resumable_status(uri, total, declared)
    if on_commit:
        on_commit(total)
    return meta


def check_drive_copy(meta, size, sha256_hex, md5_hex=None):
    # Drive's own checksums must equal the local ones.
    if int(meta.get("size", -1)) != size:
        raise RuntimeError(f"Drive file {meta.get('name')} is {meta.get('size')} bytes, expected {size:,}")
    if meta.get("sha256Checksum"):
        if meta["sha256Checksum"] != sha256_hex:
            raise RuntimeError(f"Drive file {meta['name']} sha256 {meta['sha256Checksum']} != local {sha256_hex}")
    elif md5_hex and meta.get("md5Checksum"):
        if meta["md5Checksum"] != md5_hex:
            raise RuntimeError(f"Drive file {meta['name']} md5 {meta['md5Checksum']} != local {md5_hex}")
    else:
        raise RuntimeError(f"Drive returned no checksum for {meta.get('name')}")


def drive_put_bytes(parent_id, name, body):
    # Replace any same-named files in parent_id with one file holding body; verified.
    for stale in drive_children(parent_id, name):
        drive_delete(stale["id"])
    uri = drive_resumable_start(parent_id, name, len(body))
    meta = drive_resumable_upload(uri, len(body), lambda offset, n: body[offset:offset + n])
    check_drive_copy(meta, len(body), hashlib.sha256(body).hexdigest(), hashlib.md5(body).hexdigest())
    return meta


def drive_get_bytes(parent_id, name):
    hits = drive_children(parent_id, name)
    if len(hits) > 1:
        raise RuntimeError(f"{len(hits)} files named {name!r} in Drive folder {parent_id}")
    return drive_download(hits[0]["id"]) if hits else None


def resolve_drive_folder(mounted_path, folder_id=""):
    # Drive id of a mounted folder: explicit id, the mount's xattr, or an API walk from My Drive.
    if folder_id:
        return folder_id, "DRIVE_DATASET_FOLDER_ID"
    try:
        return os.getxattr(mounted_path, "user.drive.id").decode().strip(), "mount attribute"
    except (OSError, AttributeError):
        pass
    parts = Path(mounted_path).relative_to("/content/drive/MyDrive").parts
    parent = "root"
    for part in parts:
        hits = [f for f in drive_children(parent, part) if f["mimeType"] in (FOLDER_MIME, SHORTCUT_MIME)]
        if len(hits) != 1:
            raise RuntimeError(f"{len(hits)} folders named {part!r} on the way to {mounted_path}; create the "
                               "folder (or set DRIVE_DATASET_FOLDER_ID to its id from the folder's URL)")
        parent = hits[0]["shortcutDetails"]["targetId"] if hits[0]["mimeType"] == SHORTCUT_MIME else hits[0]["id"]
    return parent, "Drive API path walk"


if IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
drive_http = drive_authorized_session()
_about = drive_request("GET", "/drive/v3/about", params={"fields": "user(emailAddress)"}).json()
dest_folder_id, _how = resolve_drive_folder(DRIVE_DEST_DIR, DRIVE_DEST_FOLDER_ID)
_folder = drive_meta(dest_folder_id)
if not _folder or _folder["mimeType"] != FOLDER_MIME:
    raise RuntimeError(f"Drive id {dest_folder_id} ({_how}) is not a folder this account can see")
if not _folder.get("capabilities", {}).get("canAddChildren", False):
    raise RuntimeError(f"Account {_about['user']['emailAddress']} cannot add files to Drive folder {_folder['name']!r}")
_free = drive_free_bytes()
print(f"Google Drive: {_about['user']['emailAddress']} | destination {DRIVE_DEST_DIR} = folder {_folder['name']!r} "
      f"({dest_folder_id}, via {_how}) | free storage " + ("unlimited" if _free is None else f"{_free / 2**30:,.1f} GiB"))

def colab_secret(name):
    # An environment variable wins; otherwise the Colab secret of that name.
    if os.environ.get(name):
        return os.environ[name].strip()
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:  # secret absent, or notebook access not granted
        return None
    return value.strip() if value else None


for _name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
    _value = colab_secret(_name)
    if _value:
        os.environ[_name] = _value
os.environ["AWS_DEFAULT_REGION"] = colab_secret("AWS_DEFAULT_REGION") or "us-east-1"
if boto3.Session().get_credentials() is None:
    raise RuntimeError("No AWS credentials. In Colab open Secrets (key icon), add AWS_ACCESS_KEY_ID and "
                       "AWS_SECRET_ACCESS_KEY, turn on notebook access for both, and rerun this cell.")
try:
    _arn = boto3.Session().client("sts").get_caller_identity()["Arn"]
except Exception as exc:
    raise RuntimeError(f"The AWS key was rejected: {type(exc).__name__}: {exc}") from exc
print(f"AWS read key: {_arn}")

## 3. Copy S3 -> Drive and verify

1. Reads the S3 `.sha256` sidecar and the archive's size and ETag.
2. Uploads the archive to Drive in `DRIVE_CHUNK_BYTES` pieces, each read from S3 by byte range. Every read is pinned to the ETag, so a changed S3 object stops the copy. A dropped connection resumes from the bytes Drive reports it holds.
3. Checks Drive's size and `sha256Checksum` against the sidecar. If Drive has not reported the checksum after 10 minutes, the file is read back from Drive and hashed instead.
4. Writes the `.sha256` sidecar (byte-for-byte the S3 one, so `sha256sum -c` works) and `.source.json` next to it.

In [ ]:
def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path.strip("/"):
        raise ValueError(f"Expected s3://bucket/key, got {uri!r}")
    return parsed.netloc, parsed.path.lstrip("/")


s3 = boto3.Session().client("s3")
src_bucket, src_key = split_s3_uri(S3_SOURCE)
sha_bucket, sha_key = split_s3_uri(S3_SHA256_SOURCE)
sidecar_body = s3.get_object(Bucket=sha_bucket, Key=sha_key)["Body"].read()
_fields = sidecar_body.decode("utf-8").strip().split()
if len(_fields) != 2 or _fields[1].lstrip("*") != ARCHIVE_NAME:
    raise ValueError(f"S3 sidecar must name {ARCHIVE_NAME}: {sidecar_body[:200]!r}")
expected_sha256 = _fields[0].lower()
if len(expected_sha256) != 64 or any(c not in "0123456789abcdef" for c in expected_sha256):
    raise ValueError("S3 sidecar does not contain a valid SHA-256")
_head = s3.head_object(Bucket=src_bucket, Key=src_key)
size, etag = int(_head["ContentLength"]), _head["ETag"]
source_identity = {"uri": S3_SOURCE, "sha256_uri": S3_SHA256_SOURCE, "sha256": expected_sha256,
                   "bytes": size, "etag": etag}
print(f"S3: {size / 2**30:.2f} GiB, ETag {etag}, SHA-256 {expected_sha256}")


def drive_sha256(meta):
    # Drive's sha256Checksum (waits up to 10 min for it), else hash the file read back from Drive.
    for _ in range(20):
        meta = drive_meta(meta["id"]) or meta
        if meta.get("sha256Checksum"):
            return meta["sha256Checksum"], "Drive's sha256Checksum"
        time.sleep(30)
    digest = hashlib.sha256()
    for offset in range(0, size, DRIVE_CHUNK_BYTES):
        digest.update(drive_download(meta["id"], offset, min(DRIVE_CHUNK_BYTES, size - offset)))
    return digest.hexdigest(), "hash of the file read back from Drive"


existing = drive_children(dest_folder_id, ARCHIVE_NAME)
if len(existing) > 1:
    raise RuntimeError(f"{len(existing)} files named {ARCHIVE_NAME} in {DRIVE_DEST_DIR}; keep at most one")
if existing:
    if int(existing[0].get("size", -1)) != size or drive_sha256(existing[0])[0] != expected_sha256:
        raise RuntimeError(f"{DRIVE_DEST_DIR}/{ARCHIVE_NAME} exists but differs from S3. Move or delete it, "
                           "then rerun this cell.")
    print(f"Already on Drive and verified: {DRIVE_DEST_DIR}/{ARCHIVE_NAME}")
else:
    if _free is not None and _free < size + 2**30:
        raise RuntimeError(f"{_free / 2**30:.1f} GiB free on Drive; need {(size + 2**30) / 2**30:.1f} GiB")

    def read_s3(offset, n):
        return s3.get_object(Bucket=src_bucket, Key=src_key, IfMatch=etag,
                             Range=f"bytes={offset}-{offset + n - 1}")["Body"].read()

    t0, _reported = time.monotonic(), [-1]
    def on_commit(committed):
        step = int(10 * committed / size)
        if step > _reported[0]:
            _reported[0] = step
            elapsed = time.monotonic() - t0
            print(f"  {committed / 2**30:.2f} / {size / 2**30:.2f} GiB | {elapsed / 60:.1f} min"
                  + (f" | {committed / 2**20 / elapsed:.0f} MiB/s" if committed and elapsed else ""), flush=True)

    meta = drive_resumable_upload(drive_resumable_start(dest_folder_id, ARCHIVE_NAME, size), size, read_s3,
                                  on_commit=on_commit)
    got, how = drive_sha256(meta)
    if int(meta.get("size", size)) != size or got != expected_sha256:
        drive_delete(meta["id"])
        raise RuntimeError(f"Copy on Drive differs from S3 (sha256 {got} via {how}); it was deleted. Rerun this cell.")
    print(f"Copied and verified in {(time.monotonic() - t0) / 60:.1f} min ({how} = S3 sidecar)")

drive_put_bytes(dest_folder_id, ARCHIVE_NAME + ".sha256", sidecar_body)
drive_put_bytes(dest_folder_id, ARCHIVE_NAME + ".source.json", (json.dumps(
    {**source_identity, "copied_utc": datetime.now(timezone.utc).isoformat()}, indent=2) + "\n").encode())
print(f"Done: {DRIVE_DEST_DIR}/{ARCHIVE_NAME} (+ .sha256, .source.json)")